In [1]:
from model import alpha, self_energy
from smatrix import create_self_energy_interpolator_numba
import numpy as np
from eigenstate_solving import BZ_proj
from model import square_lattice


sigma_data = np.load("../../data/sigma_grid0f1a.npz")
kx = sigma_data["kx"]
ky = sigma_data["ky"]
sigma_grid = sigma_data["sigma_grid"]
sigma_func_period_numba = create_self_energy_interpolator_numba(
    kx, ky, sigma_grid, lattice=square_lattice
)
collective_lamb_shift = self_energy(
    0, 0, square_lattice.a, square_lattice.d, square_lattice.omega_e, alpha
).real
sigma_func_period = create_self_energy_interpolator_numba(
    kx, ky, sigma_grid, lattice=square_lattice
)


E = 2 * (square_lattice.omega_e + collective_lamb_shift) + 2


Q_para = np.array([0, 0])
Zc = 0



n_points = int(2e6) # number of grid for FFT


In [2]:
from W_state import _pole_loc


def state_classifier(r_para, p_para, E1, E, Q_para,eta, lattice, sigma_func_period):
    """Label the state as "bound" or "extended" based on the number of roots of the denominator."""
    if np.linalg.norm(r_para) + np.linalg.norm(BZ_proj(Q_para-r_para,lattice)) > E:
        raise ValueError("In plane-wave basis, the total in-plane photon momenta exceeds the total energy.")

    if np.linalg.norm(p_para) + np.linalg.norm(BZ_proj(Q_para-p_para,lattice)) > E:
        raise ValueError("In W-state label, the total in-plane photon momenta exceeds the total energy.")

    root_list =  _pole_loc(r_para, p_para, E1, E, Q_para,eta, lattice, sigma_func_period)
    q_root_list = sorted(root[0] for root in root_list)
    # If two roots are very close to each other, we treat them as a single double root.
    if len(q_root_list) == 2 and np.isclose(q_root_list[0], q_root_list[1]):
        q_root_list = [(q_root_list[0] + q_root_list[1]) / 2]

    if  len(q_root_list) == 2:
        return "extended"
    elif len(q_root_list) == 1 or len(q_root_list) == 0:
        return "bound"




In [3]:
import pandas as pd

def state_label_dataframe(r_para_values, p_para_values,lattice):
    rows = []
    for p_val in p_para_values:
        p_arr = np.asarray(p_val, dtype=float)
        for r_val in r_para_values:
            r_arr = np.asarray(r_val, dtype=float)
            E1_values = np.linspace(np.linalg.norm(p_arr),E-np.linalg.norm(BZ_proj(Q_para-p_arr,lattice)),10,endpoint=False)

            for E1 in E1_values:
                rows.append(
                    {   "r_para": tuple(r_arr),
                        "p_para": tuple(p_arr),
                        "r_x": r_arr[0],
                        "p_x": p_arr[0],
                        "E1": E1,
                        "label": state_classifier(
                            r_arr,
                            p_arr,
                            E1,
                            E,
                            Q_para,
                            eta,
                            square_lattice,
                            sigma_func_period,
                        ),
                    }
                )
    return pd.DataFrame(rows, columns=["r_para", "p_para", "r_x", "p_x", "E1", "label"])




n_r_grid_points = 30 # number of grid points along each direction in the r_para path inside the light cone
r_path = []

vertical_r_grid_points =  np.linspace(0,E/2,n_r_grid_points,endpoint=False)
diagonal_r_grid_points =  np.linspace(0,E/2/np.sqrt(2),n_r_grid_points,endpoint=False)

# go through the grid in reverse order but skip the first point (0,0) to avoid duplication
for x in vertical_r_grid_points[:1:-1]:
    r_path.append(np.array([x, 0.0]))

r_path.append(np.array([0.01, 0.01]))

for x in diagonal_r_grid_points[1:]:
    r_path.append(np.array([x, x]))

p_path = r_path.copy()  # use the same path for p_para

eta = 0.0



state_df = state_label_dataframe(r_path, p_path,square_lattice)
state_df

,r_para,p_para,r_x,p_x,E1,label
0,"(87.75509994979575, 0.0)","(87.75509994979575, 0.0)",87.755100,87.755100,87.755100,bound
1,"(87.75509994979575, 0.0)","(87.75509994979575, 0.0)",87.755100,87.755100,88.360308,extended
2,"(87.75509994979575, 0.0)","(87.75509994979575, 0.0)",87.755100,87.755100,88.965515,extended
3,"(87.75509994979575, 0.0)","(87.75509994979575, 0.0)",87.755100,87.755100,89.570723,extended
4,"(87.75509994979575, 0.0)","(87.75509994979575, 0.0)",87.755100,87.755100,90.175930,extended
...,...,...,...,...,...,...
33635,"(62.05222625820383, 62.05222625820383)","(62.05222625820383, 62.05222625820383)",62.052226,62.052226,90.781138,bound
33636,"(62.05222625820383, 62.05222625820383)","(62.05222625820383, 62.05222625820383)",62.052226,62.052226,91.386345,extended
33637,"(62.05222625820383, 62.05222625820383)","(62.05222625820383, 62.05222625820383)",62.052226,62.052226,91.991553,extended
33638,"(62.05222625820383, 62.05222625820383)","(62.05222625820383, 62.05222625820383)",62.052226,62.052226,92.596761,extended


In [4]:
import plotly.graph_objects as go

fig = go.Figure()

for label, color in [("bound", "blue"), ("extended", "red")]:
    mask = state_df["label"] == label
    fig.add_trace(
        go.Scatter3d(
            x=state_df.loc[mask, "r_x"],
            y=state_df.loc[mask, "p_x"],
            z=state_df.loc[mask, "E1"],
            mode="markers",
            marker=dict(size=3, color=color, opacity=0.7),
            name=label,
        )
    )

fig.update_layout(
    scene=dict(
        xaxis_title="r_x",
        yaxis_title="p_x",
        zaxis_title="E_1",
    ),
    legend_title_text="state",
    width=900,
    height=700,
)

fig.show()


In [7]:
target_E1 = 90.0
E1_window = 1

slice_mask = (state_df["E1"] - target_E1).abs() <= E1_window
slice_df = state_df.loc[slice_mask].copy()

if slice_df.empty:
    nearest_E1 = state_df.loc[(state_df["E1"] - target_E1).abs().idxmin(), "E1"]
    slice_df = state_df.loc[np.isclose(state_df["E1"], nearest_E1)].copy()
    title = f"Nearest available E_1 slice to {target_E1:g}: E_1 = {nearest_E1:.3f}"
else:
    title = f"E_1 slice: {target_E1 - E1_window:g} <= E_1 <= {target_E1 + E1_window:g}"

print(f"Plotting {len(slice_df)} points with E_1 from {slice_df['E1'].min():.3f} to {slice_df['E1'].max():.3f}")

fig_slice = go.Figure()

for label, color in [("bound", "blue"), ("extended", "red")]:
    mask = slice_df["label"] == label
    fig_slice.add_trace(
        go.Scatter(
            x=slice_df.loc[mask, "r_x"],
            y=slice_df.loc[mask, "p_x"],
            mode="markers",
            marker=dict(size=8, color=color, opacity=0.8),
            name=label,
            text=slice_df.loc[mask, "E1"].map(lambda value: f"E_1={value:.3f}"),
            hovertemplate="r_x=%{x:.3f}<br>p_x=%{y:.3f}<br>%{text}<extra>%{fullData.name}</extra>",
        )
    )

fig_slice.update_layout(
    title=title,
    xaxis_title="r_x",
    yaxis_title="p_x",
    legend_title_text="state",
    width=700,
    height=600,
)

fig_slice.show()


Plotting 3712 points with E_1 from 89.571 to 90.781
